In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Veri setini yükleme
df = pd.read_csv('../scraping/data/encoded.csv')

# Veriye ilk bakış
display(df.head())
display(df.info())

,fiyat,metrekare_brut,metrekare_net,bina_yasi,kat,kat_sayisi,banyo_sayisi,ilce,mahalle,kat_orani,...,isitma_Klima,isitma_Kombi,isitma_Kombi Doğalgaz,isitma_Merkezi,isitma_Merkezi (Pay Öl...,isitma_Merkezi (Pay Ölçer),isitma_Merkezi Doğalgaz,isitma_Soba,isitma_VRV,isitma_Yerden Isıtma
0,10450000.0,120.0,105.0,25.0,3.0,4.0,1.0,7.737870e+06,7.790845e+06,0.750000,...,0,0,1,0,0,0,0,0,0,0
1,7599000.0,120.0,100.0,7.5,2.0,5.0,1.0,7.391275e+06,7.329623e+06,0.400000,...,0,0,1,0,0,0,0,0,0,0
2,4450000.0,120.0,115.0,7.5,1.0,4.0,1.0,7.100478e+06,6.668028e+06,0.250000,...,0,0,0,0,0,0,0,0,0,0
3,5750000.0,120.0,115.0,0.0,2.0,4.0,2.0,7.100478e+06,6.668028e+06,0.500000,...,0,0,0,0,0,0,0,0,0,0
4,8500000.0,115.0,95.0,25.0,6.0,11.0,1.0,7.100478e+06,6.778087e+06,0.545455,...,0,0,1,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 12127 entries, 0 to 12126
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   fiyat                       12127 non-null  float64
 1   metrekare_brut              12127 non-null  float64
 2   metrekare_net               12127 non-null  float64
 3   bina_yasi                   12127 non-null  float64
 4   kat                         12127 non-null  float64
 5   kat_sayisi                  12127 non-null  float64
 6   banyo_sayisi                12127 non-null  float64
 7   ilce                        12127 non-null  float64
 8   mahalle                     12127 non-null  float64
 9   kat_orani                   12127 non-null  float64
 10  metrekare_farki             12127 non-null  float64
 11  metrekare_verimliligi       12127 non-null  float64
 12  oda_sayisi                  12127 non-null  float64
 13  salon_sayisi                12127 non-null

None

In [2]:
# Hedef değişkeni (y) boyutunu ayarlıyoruz (Scaler'ın kabul etmesi için 2 boyutlu olmalı)
X = df.drop('fiyat', axis=1)
y = df['fiyat'].values.reshape(-1, 1)

# Eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Modellerin bulunduğu ana dizin path'i
base_models_dir = "../backend/ml-infra/models"

# 2. Sonuçları kaydetmek için bir liste
all_results = []

# Klasördeki model_0, model_1 gibi alt klasörleri sıralı bir şekilde dönüyoruz
# Klasör isimlerine göre sıralama yapmak için sıralı listeliyoruz
model_folders = sorted([f for f in os.listdir(base_models_dir) if f.startswith("model_")])

print(f"Toplam {len(model_folders)} model bulundu. Test işlemi başlıyor...\n")
print("-" * 60)

for folder in model_folders:
    model_path = os.path.join(base_models_dir, folder)
    
    print(f">> {folder} işleniyor...")
    
    try:
        # 3. Modele ait spesifik dosyaların yüklenmesi
        model = load_model(os.path.join(model_path, "model.keras"))
        scaler_X = joblib.load(os.path.join(model_path, "scaler_X.pkl"))
        scaler_y = joblib.load(os.path.join(model_path, "scaler_y.pkl"))
        isitma_columns = joblib.load(os.path.join(model_path, "isitma_columns.pkl"))
        target_encoder = joblib.load(os.path.join(model_path, "target_encoder.pkl"))
        
        # 4. Veriyi bu modele ait scaler ile ölçekleme
        X_test_scaled = scaler_X.transform(X_test)
        
        # 5. Modelden tahmin alma
        y_pred_scaled = model.predict(X_test_scaled, verbose=0)
        
        # 6. Tahminleri gerçek fiyat aralığına geri çevirme (Inverse Transform)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        
        # Eğer y_test zaten ham/gerçek fiyat ise direkt kullanıyoruz
        # Eğer y_test de ölçekli geldiyse: y_test_real = scaler_y.inverse_transform(y_test) yapmalısın.
        y_test_real = y_test 
        
        # 7. Metriklerin hesaplanması
        r2 = r2_score(y_test_real, y_pred)
        mae = mean_absolute_error(y_test_real, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
        
        # Sonuçları ekrana yazdırma
        print(f"   R2 Skoru: {r2:.4f}")
        print(f"   MAE     : {mae:,.2f} TL")
        print(f"   RMSE    : {rmse:,.2f} TL")
        print("-" * 60)
        
        # Sonuçları daha sonra toplu incelemek için listeye kaydetme
        all_results.append({
            "Model": folder,
            "R2": r2,
            "MAE": mae,
            "RMSE": rmse
        })
        
    except Exception as e:
        print(f"   [HATA] {folder} çalıştırılırken bir hata oluştu: {e}")
        print("-" * 60)

# 8. Tüm sonuçları toplu bir DataFrame olarak gösterme
df_results = pd.DataFrame(all_results)
df_results = df_results.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("\n### TÜM MODELLERİN PERFORMANS SIRALAMASI (R2'ye Göre) ###")
print(df_results.to_string(index=False))

Toplam 9 model bulundu. Test işlemi başlıyor...

------------------------------------------------------------
>> model_0 işleniyor...
   R2 Skoru: 0.4924
   MAE     : 936,591.32 TL
   RMSE    : 1,249,802.46 TL
------------------------------------------------------------
>> model_1 işleniyor...
   R2 Skoru: 0.5050
   MAE     : 916,187.03 TL
   RMSE    : 1,234,137.77 TL
------------------------------------------------------------
>> model_2 işleniyor...
   R2 Skoru: 0.5415
   MAE     : 802,674.14 TL
   RMSE    : 1,187,755.33 TL
------------------------------------------------------------
>> model_3 işleniyor...
   R2 Skoru: 0.4938
   MAE     : 943,059.03 TL
   RMSE    : 1,247,993.82 TL
------------------------------------------------------------
>> model_4 işleniyor...
   R2 Skoru: 0.3398
   MAE     : 1,104,506.20 TL
   RMSE    : 1,425,272.33 TL
------------------------------------------------------------
>> model_5 işleniyor...
   R2 Skoru: 0.4223
   MAE     : 1,053,857.83 TL
   RMSE   